In [ ]:
from analysis_helper import *

import xarray as xr
import geopandas as gpd
from matplotlib import cm, colors
from matplotlib.patches import Patch

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def compute_generation(ds, gdf):
    """
    Compute generation density in MWh / km²
    """
    p_nom_max = ds["p_nom_max"].to_pandas() / 1e3       # GW
    area = gdf.geometry.to_crs("ESRI:54009").area / 1e6 # km²
    p_max_pu = ds["profile"].to_pandas().sum()          # h
    return p_max_pu * p_nom_max / area


def values_to_rgba(values, cmap, norm,
                   min_alpha=1,
                   max_alpha=1):
    """
    Map values to RGBA colors with value-based transparency.
    Safe against values outside vmin/vmax.
    """
    # Normalize and clip to [0, 1]
    scaled = norm(values)
    scaled = np.clip(scaled, 0.0, 1.0)

    # Get RGBA colors from colormap
    rgba = cmap(scaled)

    # Set alpha based on clipped values
    rgba[:, 3] = min_alpha + scaled * (max_alpha - min_alpha)

    return rgba

savefig = False

# Generate Renewable Potential

In [ ]:
# User Setting
VMAX = 7.0          # <-- adjust this (GWh / km²)
scenario = "decarbonize-aims-3H"

# Load geometries
file_path_geom = f"../resources/{scenario}/bus_regions/regions_"
gdf_onshore = gpd.read_file(file_path_geom + "onshore.geojson").set_index("name")
gdf_offshore = gpd.read_file(file_path_geom + "offshore.geojson").set_index("name")

# Load datasets
file_path_profile = f"../resources/{scenario}/renewable_profiles/profile_"
ds_solar = xr.open_dataset(file_path_profile + "solar.nc")
ds_onwind = xr.open_dataset(file_path_profile + "onwind.nc")
ds_offwind_ac = xr.open_dataset(file_path_profile + "offwind-ac.nc")
ds_offwind_dc = xr.open_dataset(file_path_profile + "offwind-dc.nc")

# Compute generation densities
solar = compute_generation(ds_solar, gdf_onshore)
wind_on = compute_generation(ds_onwind, gdf_onshore)
wind_off_ac = compute_generation(ds_offwind_ac, gdf_offshore)
wind_off_dc = compute_generation(ds_offwind_dc, gdf_offshore)
wind_off = wind_off_ac.add(wind_off_dc, fill_value=0)

# Attach data
gdf_onshore = gdf_onshore.copy()
gdf_offshore = gdf_offshore.copy()

gdf_onshore["solar"] = solar
gdf_onshore["wind"] = wind_on
gdf_offshore["wind"] = wind_off


# Normalization (shared scale)
norm = colors.Normalize(vmin=0.0, vmax=VMAX)

solar_cmap = cm.Reds
wind_cmap = cm.Blues

# Convert values → RGBA
gdf_onshore["solar_rgba"] = list(
    values_to_rgba(gdf_onshore["solar"].values, solar_cmap, norm)
)

gdf_onshore["wind_rgba"] = list(
    values_to_rgba(gdf_onshore["wind"].values, wind_cmap, norm)
)

gdf_offshore["wind_rgba"] = list(
    values_to_rgba(gdf_offshore["wind"].values, wind_cmap, norm)
)


# Plot
fig, (ax1, ax2) = plt.subplots(1, 2,figsize=(10, 5))
ax1.axis("off")
ax2.axis("off")

# Solar (red)
gdf_onshore.plot(
    color=gdf_onshore["solar_rgba"],
    edgecolor="grey",
    linewidth=0.3,
    ax=ax1
)

# Wind onshore (blue)
gdf_onshore.plot(
    color=gdf_onshore["wind_rgba"],
    edgecolor="grey",
    linewidth=0.3,
    ax=ax2
)

# Wind offshore (blue)
gdf_offshore.plot(
    color=gdf_offshore["wind_rgba"],
    edgecolor="grey",
    linewidth=0.3,
    ax=ax2
)

# Colorbars (same height)
sm_solar = cm.ScalarMappable(norm=norm, cmap=solar_cmap)
sm_solar.set_array([])

sm_wind = cm.ScalarMappable(norm=norm, cmap=wind_cmap)
sm_wind.set_array([])

# Solar colorbar axis
cax_solar = inset_axes(
    ax1,
    width="2.5%",
    height="80%",
    loc="center right",
    bbox_transform=ax1.transAxes,
    borderpad=0
)

# Wind colorbar axis
cax_wind = inset_axes(
    ax2,
    width="2.5%",
    height="80%",
    loc="center right",
    bbox_transform=ax2.transAxes,
    borderpad=0
)

cbar_solar = fig.colorbar(sm_solar, cax=cax_solar)
cbar_solar.set_label("Solar potential [GWh / km²]")

cbar_wind = fig.colorbar(sm_wind, cax=cax_wind)
cbar_wind.set_label("Wind potential [GWh / km²]")

#plt.title("Renewable Generation Potential (Solar & Wind)", y=0.9, x=0.6)
plt.subplots_adjust(wspace=0.1)
plt.show()

if savefig:
    fig.savefig(
        "renewable-pot.png",
        dpi=300,
        bbox_inches="tight",
    )

# Generate raw electricity network

In [ ]:
# Load network
scenario = "baseline-existing-3H"
n = pypsa.Network(f"../networks/{scenario}/elec.nc")

# Scaling factors
bus_factor = 2e4       # MW → plot size
width_factor = 5e3     # MW → line width

# Plot network
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
bus_carrier = n.generators.groupby(["bus", "carrier"])["p_nom"].sum()

n.plot(
    ax=ax,
    bus_sizes=bus_carrier / bus_factor,
    line_colors="tab:blue",
    link_colors="tab:blue",
    line_widths=n.lines["s_nom"] / width_factor,
    link_widths=n.links["p_nom"] / width_factor,
)

# Carrier color legend
bus_carrier_list = bus_carrier.groupby("carrier").sum().index
ordered_carrier = [c for c in order_carrier if c in bus_carrier_list]
remaining_cols = [c for c in bus_carrier_list if c not in order_carrier]
carrier_list = n.carriers.loc[ordered_carrier + remaining_cols,["color","nice_name"]].set_index("nice_name")

carrier_handles = [
    Patch(facecolor=color, label=carrier)
    for carrier, color in carrier_list.color.items()
]

carrier_legend = ax.legend(
    handles=carrier_handles,
    #title="Carrier",
    loc="center right",
    alignment="left",
    bbox_to_anchor=(1.0, 0.5),# (1.00, 1.00),
    frameon=False
)
ax.add_artist(carrier_legend)

legend_bus = {"sizes":[20000 / bus_factor, 10000 / bus_factor, 5000 / bus_factor],
              "labels":["20 GW", "10 GW", "5 GW"]
             }

pypsa.plot.add_legend_circles(
    ax=ax, 
    sizes = legend_bus["sizes"], 
    labels = legend_bus["labels"],
    patch_kw = {'color':'silver'},
    legend_kw = {'loc':"center right",
                 'bbox_to_anchor':(0.91, 0.88),
                 'frameon':False, 
                 'labelspacing':1,
                 'alignment':"left"
                 #'title':"Bus Capacity",
                }
)

legend_line = {"sizes":[8000 / width_factor, 4000 / width_factor, 2000 / width_factor],
               "labels":["8 GW", "4 GW", "2 GW"]
              }

pypsa.plot.add_legend_lines(
    ax=ax, 
    sizes = legend_line["sizes"], 
    labels = legend_line["labels"],
    patch_kw = {'color':'tab:blue'},
    legend_kw = {'loc':"center right",
                 'bbox_to_anchor':(0.89, 0.75),
                 'frameon':False,
                 'labelspacing':1,
                 'alignment':"left"
                 #'title':"Line Capacity",
                }
)

# Final layout
ax.coastlines()
plt.tight_layout()
plt.show()

if savefig:
    fig.savefig(
        "elec-ASEAN.png",
        dpi=300,
        bbox_inches="tight",
    )


# Recalculate Fuel Cost

In [ ]:
def calculate_fuel_cost()

    fn = "https://thedocs.worldbank.org/en/doc/18675f1d1639c7a34d463f59263ba0a2-0050012025/related/CMO-Historical-Data-Annual.xlsx"
    index_mapping = {
        "coal": 'Coal, South African',
        "gas": 'Natural gas, Europe',
        "oil": 'Crude oil, average',
    }
    
    df = pd.read_excel(fn, sheet_name="Annual Prices (Real)", header=6, index_col=0).iloc[[0, -1]]  # most recent year is 2024
    df.index = ["unit","value"]
    
    df = df.rename(columns={v: k for k, v in index_mapping.items()})[index_mapping.keys()]
    
    # Add lignite column as a third of coal
    df.loc["value","lignite"] = df.loc["value","coal"] / 3 # approximation lignite = 1/3 coal
    df.loc["unit","lignite"] = df.loc["unit","coal"]
    df = df.T
    
    # add conversion to MJ
    fn = "https://www.seai.ie/sites/default/files/data-and-insights/seai-statistics/conversion-factors/SEAI-conversion-and-emission-factors.xlsx"
    conv = pd.read_excel(fn, sheet_name="Conversion and emission factors",header=19, index_col=1)
    
    bbl_to_l = 158.9872949 # https://www.convertunits.com/from/bbl/to/liter
    MJ_to_kWh = 3.6
    
    conversion_mapping = {
        "coal": conv.loc["Bituminous coal","MJ/kg"] /1e3 * MJ_to_kWh,
        "gas": 3.4121414798969, # https://www.convertunits.com/from/megawatt-hour/to/million+Btu
        "oil": conv.loc["Crude oil","MJ/l"] / bbl_to_l,
        "lignite": conv.loc["Lignite","MJ/kg"] /1e3 * MJ_to_kWh,
    }
    
    df["to_kWh_th"] = df.index.map(conversion_mapping)
    df["USD/kWh_th"] = df["value"] * df["to_kWh_th"]

    return df